# Normalization Analysis

In [ ]:
import polars as pl
import numpy as np
from scipy.optimize import curve_fit
from pathlib import Path

In [2]:

df = pl.read_csv(Path("..") / "data" / "normalization_data.csv")

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot == 0:
        # Handles horizontal data: perfect fit if ss_res is also 0
        return 1.0 if ss_res == 0 else 0.0
    return 1 - (ss_res / ss_tot)

def lin_off(xx, a, b):
    return a * xx + b

def pow_off(xx, a, b, c):
    return a * (xx ** b) + c

def exp_off(xx, a, b, c):
    return a * np.exp(b * xx) + c

fit_funcs = {
  "Linear": lin_off, 
  "Power": pow_off, 
  "Exponential": exp_off
}



In [3]:

raw_measurements = ["Mass", "TotalLength", "RightTibiaLength", "RightFemurLength", "RightFootLength", "RightWalkingSpeed"]
computed_measurements = {
  "RightLegLength": lambda df: df["RightTibiaLength"] + df["RightFemurLength"],
  "Mass*TotalLength": lambda df: df["Mass"] * df["TotalLength"],
  "Mass*RightTibiaLength": lambda df: df["Mass"] * df["RightTibiaLength"],
  "Mass*Speed": lambda df: df["Mass"] * df["RightWalkingSpeed"],
}
measurements = raw_measurements + list(computed_measurements.keys())
moments = [col for col in df.columns if col not in measurements and col != "Subject"] 

df = df.with_columns([
    *(computed_measurements[name](df).alias(name) for name in computed_measurements)
])



In [4]:
fit_results = pl.DataFrame(schema={
  "measurement": pl.String,
  "moment": pl.String,
  "fit_type": pl.String,
  "parameters": pl.List(pl.Float64),
  "covariance": pl.List(pl.Float64),
  "r2": pl.Float64,
})

for measurement in measurements:
  x = df.select(pl.col(measurement)).to_numpy().flatten()
  mask = x != 0 # Remove any missing measurements
  x = x[mask]  
  for moment in moments:
    y = df.select(abs(pl.col(moment))).to_numpy().flatten()
    y = y[mask]
    for name, fit in fit_funcs.items():
      params, cov = curve_fit(fit, x, y, maxfev=50000)
      r2 = r2_score(y, fit(x, *params))
      fit_results = fit_results.vstack(
        pl.DataFrame({
          "measurement": measurement,
          "moment": moment,
          "fit_type": name,
          "parameters": [params.tolist()],
          "covariance": [cov.flatten().tolist()],
          "r2": r2,
        })
      )
      
# Export fit_results to CSV for plotting in chapter
linear_fit_results = fit_results.filter(pl.col("fit_type") == "Linear").to_pandas()
linear_fit_results.to_csv(Path("..") / "data" / "linear_fit_results.csv", index=False)

power_fit_results = fit_results.filter(pl.col("fit_type") == "Power").to_pandas()
power_fit_results.to_csv(Path("..") / "data" / "power_fit_results.csv", index=False)

exp_fit_results = fit_results.filter(pl.col("fit_type") == "Exponential").to_pandas()
exp_fit_results.to_csv(Path("..") / "data" / "exp_fit_results.csv", index=False)

/tmp/ipykernel_448463/2216747541.py:18: OptimizeWarning: Covariance of the parameters could not be estimated
  params, cov = curve_fit(fit, x, y, maxfev=50000)
